# Heatmaps to keypoint coordinates

The local fixture creates a Gaussian heatmap, decodes its argmax, and applies a bounded interior subpixel correction.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/21-keypoint-pose')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l21_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
import numpy as np
heatmap = module.gaussian_heatmap(32, 12.0, 18.0, sigma=2.0)
assert heatmap.shape == (32, 32) and np.isclose(heatmap.max(), 1.0)
asymmetric = np.zeros((1, 1, 8, 8), dtype=np.float32)
asymmetric[0, 0, 3, 3] = 1.0
asymmetric[0, 0, 3, 2] = 0.2
asymmetric[0, 0, 3, 4] = 0.9
asymmetric[0, 0, 2, 3] = 0.1
asymmetric[0, 0, 4, 3] = 0.8
decoded = module.numpy_heatmap_to_coords(asymmetric)
refined = module.numpy_subpixel_refine(asymmetric)
assert decoded.tolist() == [[[3.0, 3.0]]] and np.allclose(refined, [[[3.25, 3.25]]])
print({'numpy_argmax': decoded.tolist(), 'numpy_subpixel': refined.tolist()})
if module.TORCH_AVAILABLE:
    import torch
    torch_decoded = module.heatmap_to_coords(torch.from_numpy(asymmetric))
    assert torch_decoded.shape == (1, 1, 2)
    print({'torch_argmax': torch_decoded.tolist()})
else:
    print('PyTorch unavailable; NumPy argmax and subpixel decoding ran.')

A heatmap gives one peak per keypoint in this lesson. Multi-person association, 3-D lifting, and PAFs require additional modeling and are intentionally outside this artifact.